# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates a step-by-step exploration of the FAIR^2 dataset using the `mlcroissant` library. We guide you through loading the metadata, reviewing available data record sets and fields (with `@id` references), extracting and analyzing data, and visualizing relationships—all following best practices for working with Croissant-compliant datasets.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

We use the Croissant schema URL and load the dataset, printing its title and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

- Each record set, field, or column has a unique `@id`. We list them for exploration. 
- This is essential for reliable referencing, as required when using Croissant/`mlcroissant` datasets.

In [ ]:
# List all RecordSets by their @id and show their fields/columns
from mlcroissant.types.dataset import DatasetMetadata

def get_record_sets(dataset_metadata: DatasetMetadata):
    # record_sets is a list of RecordSetMetadata objects
    if not hasattr(dataset_metadata, 'record_sets'):
        return []
    return dataset_metadata.record_sets or []

record_sets = get_record_sets(metadata)

if record_sets:
    print("Record Sets found in dataset:")
    for rs in record_sets:
        print(f"- {rs.id} (name: {getattr(rs, 'name', '<no name>')})")
        if hasattr(rs, 'fields') and rs.fields:
            for field in rs.fields:
                print(f"    - Field: {field.id} (name: {getattr(field, 'name', '<no name>')})")
        if hasattr(rs, 'columns') and rs.columns:
            for col in rs.columns:
                print(f"    - Column: {col.id} (name: {getattr(col, 'name', '<no name>')})")
else:
    print("No record sets found in the dataset. Some Croissant packages only describe data files and variables, you may need to inspect 'distribution' or other metadata for details.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis using record set/field `@id`s from the overview.

*If the record sets list above is empty, you may need to directly inspect or refer to documentation/distribution sections for the data tables.*

In [ ]:
# Collect all record_set @ids
record_set_ids = [r.id for r in get_record_sets(metadata)]
dataframes = {}

if not record_set_ids:
    print("No record sets defined in the metadata. Unable to extract tabular data with 'mlcroissant'.\n",
          "Check 'distribution' metadata or documentation for direct download URLs, or file descriptions.")
else:
    print(f"Available record_set @ids:\n  {record_set_ids}")
    for record_set_id in record_set_ids:
        print(f"Extracting records for record_set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")
    # Show columns and sample for the first available record set
    first_rs_id = record_set_ids[0]
    print(f"Available columns for record_set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate basic analysis: 
- Filtering numeric fields
- Normalizing values
- Grouping by a categorical field

**All references below are made by `@id`.**
If you are unsure about field types or want to see available field `@id`s and names, revisit Section 2.

In [ ]:
# Example: EDA on first record set (update field ids as appropriate)
import numpy as np

if not record_set_ids or not dataframes:
    print("No record set data available. Cannot proceed with EDA.")
else:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Analyzing first record set: {record_set_id}")
    print("Available columns (by @id):")
    print(df.columns.tolist())
    # Select a numeric field (update the field ID based on your dataset)
    # Example placeholder: numeric_field_id = 'cr:field/log_likelihood'
    # If you know the correct @id or column name here, update accordingly
    if len(df.columns) > 0:
        # Guess a likely numeric field based on common regression output names
        possible_num = [c for c in df.columns if ('log' in c and 'likelihood' in c) or 'beta' in c or 'coef' in c or 'value' in c.lower()]
        if possible_num:
            numeric_field = possible_num[0]
            print(f"Using numeric field: {numeric_field}")
        else:
            numeric_field = df.columns[0]
            print(f"No obvious numeric field found, using first column: {numeric_field}")

        if pd.api.types.is_numeric_dtype(df[numeric_field]):
            threshold = df[numeric_field].mean()
            filtered_df = df[df[numeric_field] > threshold]
        else:
            print(f"Selected field '{numeric_field}' is not numeric, EDA steps may not apply.")
            filtered_df = df
            threshold = None

        print(f\nFiltered records where '{numeric_field}' > {threshold if threshold is not None else '[N/A]'}:")
        print(filtered_df.head())

        # Normalization (only if numeric)
        if pd.api.types.is_numeric_dtype(df[numeric_field]):
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"\nNormalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Find a possible group field (categorical, different from the numeric)
        possible_groups = [c for c in df.columns if c != numeric_field and (df[c].dtype == 'object' or df[c].dtype.name.startswith('category'))]
        if possible_groups:
            group_field = possible_groups[0]
            print(f"\nGrouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("\nNo suitable group/categorical field found for grouping.")
    else:
        print("\nNo fields/columns detected in this record set.")

## 5. Visualization
Visualize distributions and relationships between fields (by `@id`).
Here, we plot the numeric field distribution and a grouped bar/chart if appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or not dataframes:
    print("No record sets or DataFrames available for visualization.")
else:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Use the same numeric_field/group_field logic as above
    possible_num = [c for c in df.columns if ('log' in c and 'likelihood' in c) or 'beta' in c or 'coef' in c or 'value' in c.lower()]
    numeric_field = possible_num[0] if possible_num else df.columns[0]
    # Histogram of numeric field
    if pd.api.types.is_numeric_dtype(df[numeric_field]):
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='royalblue')
        plt.title(f"Distribution of {numeric_field} (@id)")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
        # Grouped bar if categorical is available
        possible_groups = [c for c in df.columns if c != numeric_field and df[c].dtype == 'object']
        if possible_groups:
            group_field = possible_groups[0]
            plt.figure(figsize=(10,4))
            sns.barplot(
                data=df[[group_field, numeric_field]].dropna().groupby(group_field).mean().reset_index(),
                x=group_field,
                y=numeric_field,
                color='seagreen'
            )
            plt.xticks(rotation=90)
            plt.title(f"Mean {numeric_field} by {group_field}")
            plt.ylabel(f"Mean {numeric_field}")
            plt.xlabel(group_field)
            plt.show()
        else:
            print("No suitable categorical field for grouped visualization.")
    else:
        print(f"Numeric visualization skipped: '{numeric_field}' is not numeric.")

## 6. Conclusion

- In this notebook, you have loaded and explored a Croissant-based dataset on rangeland management adoption predictors in Northern Kenya.
- All entities (record sets, fields, columns) were referenced and accessed via their Croissant `@id`.
- We outlined best practices for inspecting metadata, loading tables, performing initial EDA, and visualizing relationships in your data.

**Next steps**: Explore specific fields or statistical relationships relevant to your analyses. Refer to the Croissant documentation and your dataset's `@id` structure for advanced usage!